# Validación visual de `local_view` (Fase 3.b / Tier 2)

Plotea 15 CP + 15 FP elegidos random del Tier 2 train. Grilla 6x5.
Output: `paper/figures/local_view_samples.png`.

Sanity check: las curvas CP deben mostrar un dip centrado en fase=0; las FP
deben ser mayoritariamente planas o con dips desalineados.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

REPO_ROOT = Path('..').resolve()
if (REPO_ROOT / 'src').exists():
    sys.path.insert(0, str(REPO_ROOT / 'src'))
TIER2_TRAIN = REPO_ROOT / 'data' / 'splits' / 'tier2_train_tics.csv'
LOCAL_DIR = REPO_ROOT / 'data' / 'processed' / 'local'
OUT_PNG = REPO_ROOT / 'paper' / 'figures' / 'local_view_samples.png'
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)

assert TIER2_TRAIN.exists(), f'No existe {TIER2_TRAIN}. Corré scripts/make_tier2_splits.py.'
assert LOCAL_DIR.exists(), f'No existe {LOCAL_DIR}.'

In [ ]:
df = pd.read_csv(TIER2_TRAIN)
print(f'Tier 2 train: {len(df)} TICs')
print(df['label'].value_counts().to_dict())

rng = np.random.default_rng(seed=42)
cp = df[df['label'] == 1].sample(15, random_state=rng.integers(0, 1_000_000_000))
fp = df[df['label'] == 0].sample(15, random_state=rng.integers(0, 1_000_000_000))
samples = pd.concat([cp, fp], ignore_index=True)
print(f'Muestras: {len(samples)} (15 CP + 15 FP)')

In [ ]:
fig, axes = plt.subplots(6, 5, figsize=(18, 18), constrained_layout=True)
phase_axis = np.linspace(-1.0, 1.0, 201)  # eje normalizado en duraciones aprox.

for ax, (_, row) in zip(axes.flat, samples.iterrows()):
    tid = int(row['tid'])
    label = int(row['label'])
    payload = torch.load(LOCAL_DIR / f'{tid}.pt', weights_only=False)
    lv = payload['local_view'].squeeze(0).numpy()
    color = 'tab:blue' if label == 1 else 'tab:red'
    tag = 'CP' if label == 1 else 'FP'
    ax.plot(phase_axis, lv, color=color, lw=1.0)
    ax.axvline(0.0, color='gray', ls=':', lw=0.8)
    ax.set_title(f'TIC {tid} ({tag}) P={payload["period"]:.2f}d', fontsize=9)
    ax.set_ylim(np.nanmin(lv) - 0.001, np.nanmax(lv) + 0.001)
    ax.tick_params(labelsize=7)

fig.suptitle('local_view — 15 CP (azul) + 15 FP (rojo)', fontsize=14)
fig.savefig(OUT_PNG, dpi=120)
print(f'Guardado: {OUT_PNG}')
plt.show()